# 01 · Data preparation

**Goal:** turn 24 months of raw West Yorkshire police records into a clean table of personal-safety crimes in Leeds, and load the locations of pubs, bars and nightclubs.

| Step | What happens |
|---|---|
| 1 | Merge the 24 monthly police files |
| 2 | Remove records without a location |
| 3 | Keep Leeds records only (by LSOA name) |
| 4 | Keep the four personal-safety crime types |
| 5 | Load the nightlife venues from OpenStreetMap |
| 6 | Save the processed data for the analysis notebooks |

**Inputs:** `data/raw/police/`, `data/raw/leeds_social_venues.csv` (see `data/README.md`)
**Outputs:** `data/processed/leeds_safety_crimes.csv`, `data/processed/leeds_venues.csv`, summary tables in `results/tables/`

In [1]:
import sys
from pathlib import Path

# Make the project's `src` package importable from inside notebooks/
sys.path.insert(0, str(Path.cwd().resolve().parent))

import pandas as pd

from src import config, data

config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 · Merge the monthly police files

data.police.uk publishes one *street-level* CSV per force per month. Each row is one recorded crime, with its month, approximate location, LSOA (a small census area of roughly 1,500 residents) and crime type.

In [2]:
raw = data.load_police_data()
print("Months covered:", raw["Month"].min(), "to", raw["Month"].max(), f"({raw['Month'].nunique()} months)")
raw.head()

Read 24 monthly files -> 642,403 records


Months covered: 2023-01 to 2024-12 (24 months)


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,02c0b7ab66e0d508d9e6d88d2771dd3a1dd6fea69aa8cd...,2023-01,West Yorkshire Police,West Yorkshire Police,-1.553889,53.604630,On or near Huddersfield Road,E01007359,Barnsley 005C,Violence and sexual offences,Unable to prosecute suspect,NaN
1,419b348ae6977dd9432b2d37724374530ddaa23d8ba103...,2023-01,West Yorkshire Police,West Yorkshire Police,-1.553889,53.604630,On or near Huddersfield Road,E01007359,Barnsley 005C,Violence and sexual offences,Unable to prosecute suspect,NaN
2,NaN,2023-01,West Yorkshire Police,West Yorkshire Police,-1.868104,53.935428,On or near Lumb Gill Lane,E01010646,Bradford 001A,Anti-social behaviour,NaN,NaN
3,NaN,2023-01,West Yorkshire Police,West Yorkshire Police,-1.881027,53.943719,On or near Main Street,E01010646,Bradford 001A,Anti-social behaviour,NaN,NaN
4,04c76965a81915c19060c589c799b82ffe073c723b7ee9...,2023-01,West Yorkshire Police,West Yorkshire Police,-1.888151,53.945946,On or near Burns Hill,E01010647,Bradford 001B,Other theft,Investigation complete; no suspect identified,NaN


Missing values per column, to decide what cleaning is needed:

In [3]:
raw.isna().sum().to_frame("missing")

,missing
Crime ID,55305
Month,0
Reported by,0
Falls within,0
Longitude,8438
Latitude,8438
Location,0
LSOA code,8438
LSOA name,8438
Crime type,0


`Latitude`, `Longitude` and both LSOA columns are missing together for the same records: crimes the police could not geolocate. `Context` is always empty. `Crime ID` and `Last outcome category` are missing for anti-social behaviour, which isn't a recorded crime, so those rows are kept.

## Step 2 · Remove records without a location

In [4]:
crimes = data.clean_police_data(raw)
crimes.columns.tolist()

Removed 8,438 records without a location -> 633,965 remain


['Crime ID',
 'Month',
 'Longitude',
 'Latitude',
 'Location',
 'LSOA code',
 'LSOA name',
 'Crime type',
 'Last outcome category']

## Step 3 · Keep Leeds records only

The West Yorkshire file covers five districts. Every LSOA name starts with its district (e.g. `Leeds 111B`), so filtering on the name gives an exact administrative boundary. An earlier version clipped points to a bounding box instead, which wrongly kept records from neighbouring Bradford, Kirklees and Calderdale.

In [5]:
leeds = data.filter_leeds(crimes)

all_types = (leeds["Crime type"].value_counts().rename_axis("Crime type").reset_index(name="Records"))
all_types["Share"] = (all_types["Records"] / all_types["Records"].sum() * 100).round(1)
all_types.to_csv(config.TABLES_DIR / "leeds_all_crime_types.csv", index=False)
all_types

Leeds records: 230,463 (36.4% of West Yorkshire)


,Crime type,Records,Share
0,Violence and sexual offences,87350,37.9
1,Public order,21604,9.4
2,Shoplifting,20455,8.9
3,Anti-social behaviour,18186,7.9
4,Criminal damage and arson,17119,7.4
5,Other theft,15734,6.8
6,Vehicle crime,14447,6.3
7,Burglary,13229,5.7
8,Drugs,6212,2.7
9,Other crime,5921,2.6


## Step 4 · Keep the four personal-safety crime types

The question is about risk to people on a night out, so the analysis keeps the categories that directly threaten personal safety: **violence and sexual offences, anti-social behaviour, robbery and theft from the person**. Property crimes such as burglary, vehicle crime and shoplifting are excluded.

In [6]:
safety = data.filter_safety_crimes(leeds)

breakdown = (safety["Crime type"].value_counts().rename_axis("Crime type").reset_index(name="Incidents"))
breakdown["Share (%)"] = (breakdown["Incidents"] / breakdown["Incidents"].sum() * 100).round(1)
breakdown.to_csv(config.TABLES_DIR / "crime_type_breakdown.csv", index=False)
breakdown

Safety-related records: 111,549 (48.4% of Leeds crime)

,Crime type,Incidents,Share (%)
0,Violence and sexual offences,87350,78.3
1,Anti-social behaviour,18186,16.3
2,Robbery,3088,2.8
3,Theft from the person,2925,2.6


The ten LSOAs with the most safety-related incidents:

In [7]:
top_lsoas = (safety["LSOA name"].value_counts().head(10).rename_axis("LSOA").reset_index(name="Incidents"))
top_lsoas["Share of Leeds (%)"] = (top_lsoas["Incidents"] / len(safety) * 100).round(1)
top_lsoas.to_csv(config.TABLES_DIR / "top_lsoas.csv", index=False)
top_lsoas

,LSOA,Incidents,Share of Leeds (%)
0,Leeds 111B,6160,5.5
1,Leeds 111C,2090,1.9
2,Leeds 111E,1147,1.0
3,Leeds 111A,1062,1.0
4,Leeds 064F,952,0.9
5,Leeds 053C,784,0.7
6,Leeds 035D,689,0.6
7,Leeds 090E,642,0.6
8,Leeds 085B,624,0.6
9,Leeds 086A,621,0.6


Incidents are highly concentrated: Leeds 111B, in the city centre, has almost three times as many as the next LSOA.

## Step 5 · Load nightlife venues

Pubs, bars and nightclubs were downloaded from OpenStreetMap with the Overpass API (`amenity=pub|bar|nightclub` inside the Leeds district boundary). The saved extract is used so the results are reproducible. Call `data.load_venues(refresh=True)` to download today's data instead, but the counts will differ slightly.

In [8]:
venues = data.load_venues()
venue_types = venues["amenity_type"].value_counts().rename_axis("Venue type").reset_index(name="Venues")
venue_types["Share (%)"] = (venue_types["Venues"] / len(venues) * 100).round(1)
venue_types.to_csv(config.TABLES_DIR / "venue_types.csv", index=False)
venue_types

Loaded 715 venues from leeds_social_venues.csv


,Venue type,Venues,Share (%)
0,pub,524,73.3
1,bar,177,24.8
2,nightclub,14,2.0


## Step 6 · Save the processed data

In [9]:
safety.to_csv(config.CRIMES_CLEAN, index=False)
venues.to_csv(config.VENUES_CLEAN, index=False)

funnel = pd.DataFrame({
    "Stage": ["Raw West Yorkshire records", "With a location", "In Leeds", "Personal-safety crime types"],
    "Records": [len(raw), len(crimes), len(leeds), len(safety)],
})
funnel.to_csv(config.TABLES_DIR / "data_funnel.csv", index=False)
funnel.style.format({"Records": "{:,}"})

,Stage,Records
0,Raw West Yorkshire records,"642,403"
1,With a location,"633,965"
2,In Leeds,"230,463"
3,Personal-safety crime types,"111,549"


**Summary:** 642,403 raw records → 111,549 personal-safety incidents in Leeds, plus 715 nightlife venues. Next: [02_hotspot_mapping.ipynb](02_hotspot_mapping.ipynb).